In [52]:
import json
import pandas as pd
from tqdm import tqdm

In [53]:
filepath = "/kaggle/input/datasets/midnightfirefly/redrob-ai-track-1-candidate-resume/dataset/candidates.jsonl"
candidates = []

with open(filepath, "r") as file:
    for line in tqdm(file):
        candidates.append(json.loads(line))

print("Total Candidates:", len(candidates))

100000it [00:07, 12866.66it/s]

Total Candidates: 100000


In [54]:
title_counts = {}
for candidate in candidates:
    title = candidate["profile"]["current_title"]
    title_counts[title] = title_counts.get(title, 0) + 1

In [55]:
title_df = pd.DataFrame(title_counts.items(), columns=["Title", "Count"])

title_df.sort_values("Count")

,Title,Count
46,Lead AI Engineer,3
36,Senior AI Engineer,4
44,Senior Applied Scientist,4
37,Senior NLP Engineer,6
40,Senior Machine Learning Engineer,6
45,Staff Machine Learning Engineer,6
39,NLP Engineer,14
38,Senior Data Scientist,19
42,AI Engineer,21
41,Applied ML Engineer,23


In [56]:
def get_experience_diff(candidate):
    claimed_exp = candidate["profile"]["years_of_experience"]
    actual_exp = sum(job["duration_months"] for job in candidate["career_history"]) / 12
    return round(abs(claimed_exp - actual_exp),1)

In [57]:
diffs = []
suspicious_candidates = []

for candidate in candidates:
    diffs.append(get_experience_diff(candidate))
    if get_experience_diff(candidate) > 1.5:
        suspicious_candidates.append(candidate)

print("Potential Honeypots:",len(suspicious_candidates))
pd.Series(diffs).describe()

Potential Honeypots: 49


count    100000.000000
mean          0.106674
std           0.226474
min           0.000000
25%           0.000000
50%           0.100000
75%           0.200000
max          15.400000
dtype: float64

In [58]:
location_counts = {}

for candidate in candidates:
    location = candidate["profile"]["location"]
    location_counts[location] = location_counts.get(location,0) + 1

In [59]:
location_df = pd.DataFrame(location_counts.items(),columns=["Location", "Count"])

location_df.sort_values("Count",ascending=False)

,Location,Count
13,"Bhubaneswar, Odisha",4321
8,"Hyderabad, Telangana",4283
5,"Noida, Uttar Pradesh",4283
27,"Jaipur, Rajasthan",4268
12,"Bangalore, Karnataka",4238
17,"Kolkata, West Bengal",4230
22,"Indore, Madhya Pradesh",4198
18,"Pune, Maharashtra",4186
1,"Chennai, Tamil Nadu",4164
20,"Delhi, Delhi",4161


In [60]:
jd_data = {
    "target_locations": [
        "Pune",
        "Noida"
    ]
}

states_and_union_territories = [
    "Andhra Pradesh",
    "Arunachal Pradesh",
    "Assam",
    "Bihar",
    "Chhattisgarh",
    "Goa",
    "Gujarat",
    "Haryana",
    "Himachal Pradesh",
    "Jharkhand",
    "Karnataka",
    "Kerala",
    "Madhya Pradesh",
    "Maharashtra",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Nagaland",
    "Odisha",
    "Punjab",
    "Rajasthan",
    "Sikkim",
    "Tamil Nadu",
    "Telangana",
    "Tripura",
    "Uttar Pradesh",
    "Uttarakhand",
    "West Bengal",
    "Delhi"
]

current_title_keywords = [
    "AI",
    "ML",
    "Machine Learning",
    "Data Scientist",
    "Applied Scientist",
    "AI Specialist",
    "AI Research",
    "NLP",
    "Search",
    "Retrieval",
    "Ranking",
    "Recommendation",
    "Data Engineer",
    "Senior Data Engineer",
]

In [61]:
class HardFilter:
    def __init__(self):
        pass
        self.filter_list: list[dict] = []
        self.allowed_current_title_filters = current_title_keywords

    def add_candidate(self, candidate: dict):
        self.filter_list.append(candidate)

    def __current_title_filter(self, candidate) -> bool:
        for keyword in self.allowed_current_title_filters:
            if keyword in candidate["profile"]["current_title"]:
                return True
        return False

    def filter_on_current_title(self):
        self.filter_list = filter(
            lambda c: self.__current_title_filter(
                c), self.filter_list
        )

    def __yoe_mismatch_filter(self, candidate):
        total_exp = candidate["profile"]["years_of_experience"]
        exp_sum = sum(career["duration_months"] for career in candidate["career_history"]) / 12
        if abs(total_exp - exp_sum) > 1.5:
            return False
        return True

    def filter_on_years_of_experience_mismatch(self):
        self.filter_list = filter(
            lambda candidate: self.__yoe_mismatch_filter(
                candidate), self.filter_list
        )

    def __location_filter(self, candidate) -> bool:
        for target_location in jd_data["target_locations"]:
            sp = candidate["profile"]["location"].split(",")
            location_state = (sp[1] if len(sp) > 1 else sp[0]).strip()
            if target_location in candidate["profile"]["location"] or (
                target_location not in candidate["profile"]["location"]
                and location_state in states_and_union_territories
            ):
                candidate["__rank_meta"] = {}
                if candidate["redrob_signals"]["willing_to_relocate"]:
                    candidate["__rank_meta"]["location_boost"] = True
                else:
                    candidate["__rank_meta"]["location_boost"] = False
                return True
                break
        return False

    def filter_on_location(self):
        self.filter_list = filter(
            lambda candidate: self.__location_filter(
                candidate), self.filter_list
        )

    def get_filtered_list(self) -> list:
        return list(self.filter_list)

In [62]:
hf = HardFilter()

for candidate in candidates:
    hf.add_candidate(candidate)

In [63]:
analysis = {"Original Pool": len(candidates)}

hf.filter_on_current_title()
hf.filter_list = list(hf.filter_list)

analysis["After Title Filter"] = len(hf.filter_list)

hf.filter_on_years_of_experience_mismatch()
hf.filter_list = list(hf.filter_list)

analysis["After Experience Filter"] = len(hf.filter_list)

hf.filter_on_location()
hf.filter_list = list(hf.filter_list)

analysis["After Location Filter"] = len(hf.filter_list)

analysis_df = pd.DataFrame(analysis.items(),columns=["Stage", "Candidates"])

analysis_df["Removed"] = (analysis_df["Candidates"].shift(1) - analysis_df["Candidates"])

analysis_df

,Stage,Candidates,Removed
0,Original Pool,100000,NaN
1,After Title Filter,2478,97522.0
2,After Experience Filter,2464,14.0
3,After Location Filter,1921,543.0


In [64]:
filtered_candidates = hf.get_filtered_list()

print("Final Candidate Pool:",len(filtered_candidates))

Final Candidate Pool: 1921
